In [1]:
import pandas as pd

In [2]:
import ee
import geemap

# ---- 1. Authenticate (first run opens a browser — sign in once) ----
ee.Authenticate()
ee.Initialize(project='my-first-project1-498804')   # <-- your GEE project id, see note below

# ---- 2. Load YOUR shapefile directly ----
shp_path = r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp"
lahore_fc = geemap.shp_to_ee(shp_path)
lahore = lahore_fc.geometry()

# ---- 3. VIIRS night-time lights: 2025 median, clipped to your boundary ----
ntl = (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
       .filterDate('2025-01-01', '2026-01-01')
       .select('avg_rad')
       .median()
       .clip(lahore))

# ---- 4. Interactive map (shows in notebook) ----
Map = geemap.Map()
Map.centerObject(lahore, 10)
vis = {'min': 0, 'max': 60,
       'palette': ['000000', '3a2c5f', 'd97706', 'ffd166', 'ffffff']}
Map.addLayer(ntl, vis, 'NTL 2025')
Map.addLayer(ee.Image().paint(lahore, 1, 2), {'palette': ['E3A93C']}, 'District boundary')
Map

# ---- 5. Total radiance inside your district (number for your paper) ----
stats = ntl.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=lahore,
    scale=500,
    maxPixels=1e9
).getInfo()
print('Total radiance inside Lahore District:', stats)

# ---- 6. Download GeoTIFF straight to your PC ----
out_tif = r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif"
geemap.ee_export_image(
    ntl,
    filename=out_tif,
    scale=500,
    region=lahore,
    crs='EPSG:4326',
    file_per_band=False
)
print('Saved:', out_tif)

Total radiance inside Lahore District: {'avg_rad': 94585.8068709108}
Generating URL ...
Please wait ...
Data downloaded to E:\economic_wealth_dashboard\lahore_ntl_2025.tif
Saved: E:\economic_wealth_dashboard\lahore_ntl_2025.tif


In [3]:
import rasterio, json

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        cells.append([round(lng, 4), round(lat, 4), round(v, 2)])

json.dump(cells, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "lit cells written to ntl_grid.json")

17052 lit cells written to ntl_grid.json


In [4]:
import rasterio, numpy as np

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    print("pixel size (deg):", src.res)
    print("shape:", a.shape, "=", a.shape[0]*a.shape[1], "pixels")
    print("NaN pixels:", int(np.isnan(a).sum()))
    print("nodata value:", src.nodata)
    print("min:", np.nanmin(a), "max:", np.nanmax(a))
    print("finite pixels > 0.5:", int((np.nan_to_num(a) > 0.5).sum()))

pixel size (deg): (0.004491576420597608, 0.004491576420597608)
shape: (116, 147) = 17052 pixels
NaN pixels: 0
nodata value: None
min: 0.705 max: 86.895004
finite pixels > 0.5: 17052


In [5]:
import rasterio, json, math
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = prep(shp.geometry.union_all())

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform
    px = abs(src.res[0])
    lng0, lat0 = tr * (0, 0)          # top-left corner of the raster

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if not math.isfinite(v) or v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        if not district.contains(Point(lng, lat)):   # keep ONLY inside your boundary
            continue
        cells.append([c, r, round(v, 2)])

out = {"lng0": lng0, "lat0": lat0, "px": px, "cells": cells}
json.dump(out, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "cells inside district | px:", round(px, 6))

8426 cells inside district | px: 0.004492


In [6]:
import rasterio, json, math
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = prep(shp.geometry.union_all())

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform
    px = abs(src.res[0])
    lng0, lat0 = tr * (0, 0)

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if not math.isfinite(v) or v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        if not district.contains(Point(lng, lat)):
            continue
        cells.append([c, r, round(v, 2)])

out = {"lng0": lng0, "lat0": lat0, "px": px, "cells": cells}
json.dump(out, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "cells inside district")

8426 cells inside district
